# 📦 Supply Chain EDA — 3 Year Analysis
**Project:** Supply Chain Optimizer | 3PL Logistics Intelligence

**Objective:** Explore 3 years of shipment data to identify delay patterns, route inefficiencies, and warehouse bottlenecks.

**Dataset:** 14,000+ synthetic shipment records (2022–2024) across 10 South African routes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Dark theme setup
plt.rcParams.update({
    'figure.facecolor':  '#060a0f',
    'axes.facecolor':    '#0a0e14',
    'axes.edgecolor':    '#1a2535',
    'axes.labelcolor':   '#e2e8f0',
    'text.color':        '#e2e8f0',
    'xtick.color':       '#4a5568',
    'ytick.color':       '#4a5568',
    'grid.color':        '#0f1820',
    'grid.linestyle':    '--',
    'figure.titlesize':  16,
    'axes.titlesize':    13,
    'axes.titlecolor':   '#e2e8f0',
    'font.family':       'monospace',
})

ACCENT  = '#e8770a'
GREEN   = '#4ade80'
BLUE    = '#60a5fa'
RED     = '#f87171'
PURPLE  = '#a78bfa'

print('✅ Libraries loaded')

In [ ]:
# ── Load dataset
df = pd.read_csv('../data/raw/shipments_3yr.csv',
                 parse_dates=['date', 'departure_datetime', 'arrival_datetime'])

print('=' * 55)
print('  SUPPLY CHAIN DATASET — OVERVIEW')
print('=' * 55)
print(f'  Shape:           {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Date range:      {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Total shipments: {len(df):,}')
print(f'  Delay rate:      {df["is_delayed"].mean()*100:.1f}%')
print(f'  Avg transit hrs: {df["actual_hours"].mean():.2f}')
print(f'  Avg delay hrs:   {df["delay_hours"].mean():.2f}')
print(f'  Total revenue:   ${df["freight_cost_usd"].sum():,.0f}')
print(f'  Unique routes:   {df["route_id"].nunique()}')
print(f'  Unique drivers:  {df["driver_id"].nunique()}')
print(f'  Unique vehicles: {df["vehicle_id"].nunique()}')
print(f'  Cargo types:     {df["cargo_type"].nunique()}')
print('=' * 55)
df.head()

In [ ]:
# ── Data types and nulls
print('Column Info:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ── Statistical summary of key numeric columns
cols = ['actual_hours', 'delay_hours', 'weather_score',
        'traffic_index', 'weight_kg', 'freight_cost_usd']
df[cols].describe().round(3)

## 1. Delivery Performance Trend (3 Years)

In [ ]:
monthly = df.groupby(['year', 'month']).agg(
    total_shipments = ('shipment_id', 'count'),
    on_time_rate    = ('on_time',     'mean'),
    delay_rate      = ('is_delayed',  'mean'),
    avg_delay_hrs   = ('delay_hours',  'mean'),
    total_revenue   = ('freight_cost_usd', 'sum'),
).reset_index()
monthly['period'] = pd.to_datetime(
    monthly[['year','month']].assign(day=1)
)
monthly['on_time_pct'] = (monthly['on_time_rate'] * 100).round(1)
monthly['delay_pct']   = (monthly['delay_rate']   * 100).round(1)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('DELIVERY PERFORMANCE — 3 YEAR TREND', fontsize=15, color='#e2e8f0', y=1.01)

# On-time rate
axes[0].fill_between(monthly['period'], monthly['on_time_pct'],
                     alpha=0.15, color=GREEN)
axes[0].plot(monthly['period'], monthly['on_time_pct'],
             color=GREEN, linewidth=2.5, label='On-Time Rate %')
axes[0].axhline(90, color=ACCENT, linestyle='--', linewidth=1.5, label='Target 90%')
axes[0].set_ylabel('On-Time Rate (%)')
axes[0].legend()
axes[0].grid(True)
axes[0].set_title('Monthly On-Time Delivery Rate')

# Monthly shipments
axes[1].bar(monthly['period'], monthly['total_shipments'],
            color=BLUE, alpha=0.8, width=20)
axes[1].set_ylabel('Total Shipments')
axes[1].set_title('Monthly Shipment Volume')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('../docs/eda_performance_trend.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()
print('✅ Chart saved → docs/eda_performance_trend.png')

## 2. Delay Analysis by Route

In [ ]:
route_stats = df.groupby(['route_id', 'origin', 'destination', 'distance_km']).agg(
    total_shipments = ('shipment_id', 'count'),
    delay_rate      = ('is_delayed',  'mean'),
    avg_delay_hrs   = ('delay_hours',  'mean'),
    efficiency_pct  = ('on_time',     'mean'),
    total_revenue   = ('freight_cost_usd', 'sum'),
).reset_index()
route_stats['delay_pct']      = (route_stats['delay_rate']     * 100).round(1)
route_stats['efficiency_pct'] = (route_stats['efficiency_pct'] * 100).round(1)
route_stats['route_label']    = route_stats['origin'].str[:3] + '→' + route_stats['destination'].str[:3]
route_stats = route_stats.sort_values('delay_pct', ascending=False)

colors = [RED if d > 25 else ACCENT if d > 15 else GREEN for d in route_stats['delay_pct']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ROUTE PERFORMANCE ANALYSIS', fontsize=14, color='#e2e8f0')

# Delay rate by route
axes[0].barh(route_stats['route_label'], route_stats['delay_pct'],
             color=colors, edgecolor='none')
axes[0].axvline(20, color=ACCENT, linestyle='--', alpha=0.7, label='20% threshold')
axes[0].set_xlabel('Delay Rate (%)')
axes[0].set_title('Delay Rate by Route')
axes[0].legend()
axes[0].grid(True, axis='x')

# Distance vs delay scatter
scatter = axes[1].scatter(
    route_stats['distance_km'], route_stats['delay_pct'],
    s=route_stats['total_shipments'] / 3,
    c=route_stats['efficiency_pct'], cmap='RdYlGn',
    alpha=0.85, edgecolors='#060a0f', linewidth=1
)
for _, row in route_stats.iterrows():
    axes[1].annotate(row['route_label'],
                     (row['distance_km'], row['delay_pct']),
                     textcoords='offset points', xytext=(6, 4),
                     fontsize=9, color='#e2e8f0')
plt.colorbar(scatter, ax=axes[1], label='Efficiency %')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Delay Rate (%)')
axes[1].set_title('Distance vs Delay Rate (bubble = volume)')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../docs/eda_route_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()
print(route_stats[['route_label','delay_pct','efficiency_pct','total_shipments']].to_string(index=False))

## 3. Delay Heatmap — Day of Week × Hour

In [ ]:
pivot = df.groupby(['day_of_week', 'departure_hour'])['is_delayed'].mean().unstack(fill_value=0)
pivot = pivot.reindex(range(7)).fillna(0) * 100

day_labels = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#060a0f')

im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Delay Rate (%)')

ax.set_xticks(range(24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(7))
ax.set_yticklabels(day_labels)
ax.set_title('DELAY HEATMAP — DAY OF WEEK × DEPARTURE HOUR', color='#e2e8f0', pad=15)
ax.set_xlabel('Departure Hour')

# Annotate cells
for i in range(7):
    for j in range(24):
        val = pivot.values[i, j]
        ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                fontsize=7, color='black' if val > 20 else 'white')

plt.tight_layout()
plt.savefig('../docs/eda_delay_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()
print('\nPeak delay periods:')
print(f'  Worst day:  {day_labels[pivot.mean(axis=1).idxmax()]}')
print(f'  Worst hour: {pivot.mean(axis=0).idxmax():02d}:00')

## 4. Delay Root Cause Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('DELAY ROOT CAUSE ANALYSIS', fontsize=14, color='#e2e8f0')

# Weather score distribution
delayed   = df[df['is_delayed'] == 1]['weather_score']
on_time   = df[df['is_delayed'] == 0]['weather_score']
axes[0,0].hist(on_time,  bins=30, color=GREEN,  alpha=0.7, label='On-Time',  density=True)
axes[0,0].hist(delayed,  bins=30, color=RED,    alpha=0.7, label='Delayed',  density=True)
axes[0,0].set_title('Weather Score Distribution')
axes[0,0].set_xlabel('Weather Score')
axes[0,0].legend()
axes[0,0].grid(True)

# Traffic index distribution
delayed_t  = df[df['is_delayed'] == 1]['traffic_index']
on_time_t  = df[df['is_delayed'] == 0]['traffic_index']
axes[0,1].hist(on_time_t, bins=30, color=GREEN, alpha=0.7, label='On-Time', density=True)
axes[0,1].hist(delayed_t, bins=30, color=RED,   alpha=0.7, label='Delayed', density=True)
axes[0,1].set_title('Traffic Index Distribution')
axes[0,1].set_xlabel('Traffic Index')
axes[0,1].legend()
axes[0,1].grid(True)

# Delay by cargo type
cargo_delay = df.groupby('cargo_type')['is_delayed'].mean().sort_values(ascending=False) * 100
cargo_colors = [RED if v > 30 else ACCENT if v > 20 else GREEN for v in cargo_delay.values]
axes[1,0].barh(cargo_delay.index, cargo_delay.values, color=cargo_colors, edgecolor='none')
axes[1,0].set_title('Delay Rate by Cargo Type')
axes[1,0].set_xlabel('Delay Rate (%)')
axes[1,0].grid(True, axis='x')

# Driver experience vs delay
exp_delay = df.groupby('driver_experience')['is_delayed'].mean() * 100
axes[1,1].scatter(exp_delay.index, exp_delay.values, color=ACCENT, s=60, alpha=0.8)
z = np.polyfit(exp_delay.index, exp_delay.values, 1)
p = np.poly1d(z)
axes[1,1].plot(exp_delay.index, p(exp_delay.index), color=RED, linewidth=2,
               linestyle='--', label='Trend')
axes[1,1].set_title('Driver Experience vs Delay Rate')
axes[1,1].set_xlabel('Experience (Years)')
axes[1,1].set_ylabel('Delay Rate (%)')
axes[1,1].legend()
axes[1,1].grid(True)

plt.tight_layout()
plt.savefig('../docs/eda_root_cause.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()

## 5. Feature Correlation Matrix

In [ ]:
features = ['weather_score', 'traffic_index', 'route_complexity', 'driver_experience',
            'driver_rating', 'departure_hour', 'day_of_week', 'is_holiday',
            'is_weekend', 'weight_kg', 'distance_km', 'planned_hours', 'is_delayed']

corr = df[features].corr()

fig, ax = plt.subplots(figsize=(13, 10))
fig.patch.set_facecolor('#060a0f')

mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, linecolor='#060a0f',
            cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})

ax.set_title('FEATURE CORRELATION MATRIX', color='#e2e8f0', pad=20, fontsize=14)
ax.tick_params(colors='#94a3b8')

plt.tight_layout()
plt.savefig('../docs/eda_correlation.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()

# Top correlations with target
print('\nTop correlations with is_delayed:')
print(corr['is_delayed'].drop('is_delayed').sort_values(ascending=False).round(4))

## 6. Revenue & Business KPIs

In [ ]:
yearly = df.groupby('year').agg(
    total_shipments = ('shipment_id',       'count'),
    on_time_rate    = ('on_time',           'mean'),
    total_revenue   = ('freight_cost_usd',  'sum'),
    avg_delay_hrs   = ('delay_hours',        'mean'),
    avg_freight     = ('freight_cost_usd',  'mean'),
).reset_index()
yearly['on_time_pct'] = (yearly['on_time_rate'] * 100).round(1)

print('=' * 60)
print('  YEARLY BUSINESS SUMMARY')
print('=' * 60)
for _, row in yearly.iterrows():
    print(f"\n  {int(row['year'])}")
    print(f"    Shipments:    {int(row['total_shipments']):,}")
    print(f"    On-Time:      {row['on_time_pct']}%")
    print(f"    Revenue:      ${row['total_revenue']:,.0f}")
    print(f"    Avg Delay:    {row['avg_delay_hrs']:.2f} hrs")
    print(f"    Avg Freight:  ${row['avg_freight']:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('YEARLY BUSINESS KPIs', fontsize=14, color='#e2e8f0')

axes[0].bar(yearly['year'], yearly['total_revenue']/1000,
            color=[ACCENT, BLUE, GREEN], edgecolor='none')
axes[0].set_title('Total Revenue (K USD)')
axes[0].set_ylabel('Revenue ($K)')
axes[0].grid(True, axis='y')

axes[1].bar(yearly['year'], yearly['on_time_pct'],
            color=[RED if v < 80 else ACCENT if v < 88 else GREEN for v in yearly['on_time_pct']],
            edgecolor='none')
axes[1].axhline(90, color=ACCENT, linestyle='--', label='Target 90%')
axes[1].set_title('On-Time Rate (%)')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].grid(True, axis='y')

axes[2].bar(yearly['year'], yearly['avg_delay_hrs'],
            color=RED, edgecolor='none', alpha=0.85)
axes[2].set_title('Avg Delay Hours')
axes[2].set_ylabel('Hours')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.savefig('../docs/eda_business_kpis.png', dpi=150, bbox_inches='tight',
            facecolor='#060a0f')
plt.show()

## 7. Key Findings Summary

In [ ]:
worst_route = route_stats.iloc[0]
best_route  = route_stats.iloc[-1]
peak_hour   = df.groupby('departure_hour')['is_delayed'].mean().idxmax()
worst_cargo = df.groupby('cargo_type')['is_delayed'].mean().idxmax()

print('=' * 60)
print('  KEY FINDINGS FROM EDA')
print('=' * 60)
print(f"\n  Overall delay rate:     {df['is_delayed'].mean()*100:.1f}%")
print(f"  Worst route:            {worst_route['route_label']} ({worst_route['delay_pct']}% delay)")
print(f"  Best route:             {best_route['route_label']} ({best_route['delay_pct']}% delay)")
print(f"  Peak delay hour:        {peak_hour:02d}:00")
print(f"  Most delayed cargo:     {worst_cargo}")
print(f"  Weather corr w/ delay:  {df['weather_score'].corr(df['is_delayed']):.3f}")
print(f"  Traffic corr w/ delay:  {df['traffic_index'].corr(df['is_delayed']):.3f}")
print(f"\n  → Recommend ML model to predict delays using weather + traffic")
print(f"  → Focus optimisation on: {worst_route['route_label']} route")
print(f"  → Avoid scheduling {worst_cargo} during peak hours")
print('=' * 60)